# Defter 2 — Teori gömülü model, serbest ağ ve eğitim
**Embedding tree growth theory into AI** · Prof. Dr. İlker Ercanlı

**Önce:** Defter 1 (Sürüm 2) çalışmış ve `ITRDB_AI/TENSOR/veri_v2.npz` oluşmuş olmalı.
**Çalışma zamanı:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**.

Defter bir **iş kuyruğu** çalıştırır: her iş = (test bölünmesi, model, tohum).
Biten işler `ITRDB_AI/SONUC/` altına yazılır ve tekrar çalıştırıldığında atlanır.
Colab bağlantısı koparsa yeniden "Tümünü çalıştır" demeniz yeterli; yarım kalan iş son epoch'tan devam eder.

**SON ÇALIŞTIRMA (21 Ekim 2026) — Analiz Planı'na göre.** Veri: `veri_v2.npz` (14 hatalı alan çıkarıldı, bölgeler koordinata göre,
yetişme ortamı değişkenleri). Sonuçlar `ITRDB_AI/SONUC_SON/` klasörüne yazılır; eski SONUC klasörüne dokunulmaz.
Model adındaki `_ortam` eki: toprak, topoğrafya, biyom, ekolojik bölge ve Köppen değişkenleri girdi olarak kullanılır.

Modeller (Methods denklemleri 1–4):
* `teori` — BAI = G(B)·C·exp(u); G Richards potansiyeli, C sınırlayıcı faktör (min) ve monoton yanıtlar, gizil başlangıç yarıçapı
* `serbest` — aynı girdiler, aynı kapasite, yapı yok (zamansal evrişim + MLP)
* `abl_G`, `abl_min`, `abl_mono`, `abl_r0` — teori modelinden birer bileşen çıkarılmış hâller (ablasyon)

In [ ]:
# --- Kurulum ---
import os, json, time, math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ANA = '/content/drive/MyDrive/ITRDB_AI'
except ImportError:
    ANA = os.environ.get('ITRDB_AI', '.')
SONUC = os.path.join(ANA, 'SONUC_SON'); os.makedirs(SONUC, exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Cihaz:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '(GPU yok: Çalışma zamanı → T4 GPU seçin)')

AYAR = dict(batch=32768, max_epoch=25, sabir=3, lr=2e-3, lr_gizil=1e-2, J=8, gizli=128, emb=8,
            takson_dusur=0.10, su_onsel=0.5, tau_onsel=0.5, sv_onsel=0.5, desc_gurultu=0.2, wd=1e-4,
            rastgele_etki=False,   # False: popülasyon modeli (ana analiz); True: ağaç/alan rastgele etkili duyarlılık analizi
            r0_bas_cm=2.0, sifir_mm=0.005, hizli_deneme=False)

In [ ]:
# --- Veri ---
d = np.load(os.path.join(ANA, 'TENSOR', 'veri_v2.npz'))
soz = json.load(open(os.path.join(ANA, 'TENSOR', 'veri_v2_sozluk.json')))
YIL0 = soz['yil0']; S = len(soz['site_code']); NG = len(soz['genus']); NS = len(soz['species'])
NSER = len(d['series_site']); NTREE = int(d['series_tree'].max()) + 1
t = lambda x, dt=None: torch.as_tensor(np.asarray(x), dtype=dt).to(DEV)

# Halkalar: 1901 halkaları çıkar (önceki yılın iklimi yok)
k = d['ring_year'] > YIL0
R_ser = t(d['ring_series'][k], torch.long); R_yil = t(d['ring_year'][k], torch.long)
R_w = t(d['ring_rw'][k], torch.float32); R_rp = t(d['ring_rprev'][k], torch.float32) / 10   # cm
ser_site = t(d['series_site'], torch.long); ser_tree = t(d['series_tree'], torch.long)
ser_zaman = t(d['series_zaman'], torch.bool)
site_genus = t(d['site_genus'], torch.long); site_species = t(d['site_species'], torch.long)
NR = len(R_w)

# Aylık iklim özellikleri [alan, ay(124*12), kanal]: T, log(1+P), M, VPD, gE
clim = d['clim']; soil = d['soil']; gE = d['gE']
NY = clim.shape[1]
ozel = np.stack([clim[..., 0], np.log1p(np.maximum(clim[..., 1], 0)), soil, clim[..., 3],
                 np.broadcast_to(gE[:, None, :], soil.shape)], -1).reshape(S, NY * 12, 5).astype(np.float32)
OZ = t(ozel); del ozel, clim
oz_ort = OZ.reshape(-1, 5).mean(0); oz_ss = OZ.reshape(-1, 5).std(0)
desc = d['desc']; DESC = t((desc - desc.mean(0)) / desc.std(0), torch.float32)
# Yetişme ortamı: 11 sürekli değişken (standartlaştırılmış) + biyom, ekolojik bölge, Köppen sınıfları (son indeks = bilinmeyen)
env = d['env']; ENV = t((env - env.mean(0)) / env.std(0), torch.float32)
site_biyom = t(d['site_biyom'], torch.long); site_eko = t(d['site_eko'], torch.long); site_kg = t(d['site_kg'], torch.long)
NB, NE, NK = soz['n_biyom'], soz['n_eko'], soz['n_kg']
KAT_GOR = {}   # bölünmeye göre eğitimde görülen sınıflar (görülmeyenler 'bilinmeyen' gömmesine gider)
ORTAM_BOYUT = ENV.shape[1] + 3 * 4

class OrtamGirdi(nn.Module):
    """Alan başına yetişme ortamı vektörü: sürekli değişkenler + biyom/ekolojik bölge/Köppen gömmeleri (her biri 4 boyut)"""
    def __init__(self):
        super().__init__()
        self.b = nn.Embedding(NB + 1, 4); self.e = nn.Embedding(NE + 1, 4); self.k = nn.Embedding(NK + 1, 4)
    def forward(self, G):
        b, e, k = G['b'], G['e'], G['k']
        if KAT_GOR:
            b = torch.where(KAT_GOR['b'][b], b, torch.full_like(b, NB)); e = torch.where(KAT_GOR['e'][e], e, torch.full_like(e, NE))
            k = torch.where(KAT_GOR['k'][k], k, torch.full_like(k, NK))
        if self.training:      # %10 rastgele maskeleme: 'bilinmeyen' gömmesi de öğrenilsin
            m = torch.rand(len(b), device=DEV) < 0.10
            b = torch.where(m, torch.full_like(b, NB), b); e = torch.where(m, torch.full_like(e, NE), e); k = torch.where(m, torch.full_like(k, NK), k)
        x = G['env'] + (AYAR['desc_gurultu'] * torch.randn_like(G['env']) if self.training else 0)
        return torch.cat([x, self.b(b), self.e(e), self.k(k)], 1)
print(f'{NR:,} halka, {NSER:,} seri, {NTREE:,} ağaç, {S} alan — veri {"GPU" if DEV=="cuda" else "CPU"} belleğinde')

def alan_girdi(s):
    """Alan düzeyi girdiler (SHAP ve ihlal analizinde gruplar halinde değiştirilebilsin diye sözlük)"""
    return dict(g=site_genus[s], sp=site_species[s], desc=DESC[s], env=ENV[s], b=site_biyom[s], e=site_eko[s], k=site_kg[s])

def girdiler(idx):
    """Halka düzeyi girdiler: 24 aylık iklim penceresi X, birikmiş yarıçap rp, seri ve alan girdileri"""
    X, s = pencere(idx); G = alan_girdi(s); G.update(X=X, rp=R_rp[idx], ser=R_ser[idx]); return G

def pencere(idx):
    """Halka idx için 24 aylık pencere (önceki yıl Ocak – bu yıl Aralık): [B, 24, 5]"""
    s = ser_site[R_ser[idx]]
    bas = (R_yil[idx] - 1 - YIL0) * 12
    ay = bas[:, None] + torch.arange(24, device=DEV)[None, :]
    return OZ[s[:, None], ay], s

In [ ]:
# --- Test bölünmeleri ---
site_bolge = np.array(soz['site_bolge']); site_kat = d['site_blok_kat']; site_takson = d['site_takson_test']
site_ic = d['site_ic_dogrulama']

def bolunme(ad):
    """Dönüş: (eğitim, doğrulama, test) halka indeksleri ve test alanlarında takson gizlensin mi"""
    site_r = ser_site[R_ser].cpu().numpy(); yil = R_yil.cpu().numpy(); zuy = ser_zaman[R_ser].cpu().numpy()
    gizle = False
    if ad == 'zaman':
        test = zuy & (yil > 1960); havuz = yil <= 1960
    elif ad.startswith('bolge:'):
        tb = np.isin(site_r, np.where(site_bolge == ad.split(':')[1])[0]); test = tb; havuz = ~tb
    elif ad.startswith('blok:'):
        tb = np.isin(site_r, np.where(site_kat == int(ad.split(':')[1]))[0]); test = tb; havuz = ~tb
    elif ad == 'takson':
        tb = np.isin(site_r, np.where(site_takson)[0]); test = tb; havuz = ~tb; gizle = True
    elif ad.startswith('iklim:'):
        # İklim dışa-genelleme: en sıcak (1901–1960 ort. sıcaklık) ya da en kurak (P/PET) %10 alan yalnız testte
        tur = ad.split(':')[1]
        x = desc[:, 2] if tur == 'sicak' else -desc[:, 5]
        tb = np.isin(site_r, np.where(x >= np.quantile(x, 0.9))[0]); test = tb; havuz = ~tb
    else:
        raise ValueError(ad)
    ic = site_ic[site_r]
    egit = np.where(havuz & ~ic)[0]; dog = np.where(havuz & ic)[0]; test = np.where(test)[0]
    return t(egit, torch.long), t(dog, torch.long), t(test, torch.long), gizle

In [ ]:
# --- Modeller ---
class AlanAgi(nn.Module):
    """z_s -> parametreler. Cins/tür gömmeleri + standartlaştırılmış alan tanımlayıcıları."""
    def __init__(self, n_cikis, emb, gizli, sutun=None, ortam=False):
        super().__init__()
        self.ortam = OrtamGirdi() if ortam else None
        self.sutun = list(range(DESC.shape[1])) if sutun is None else sutun   # saf teori: yalnız [enlem, yükselti]
        self.ge = nn.Embedding(NG + 1, emb); self.se = nn.Embedding(NS + 1, emb)   # son indeks = "bilinmeyen takson"
        self.mlp = nn.Sequential(nn.Linear(2 * emb + len(self.sutun) + (ORTAM_BOYUT if ortam else 0), gizli), nn.GELU(),
                                 nn.Linear(gizli, gizli), nn.GELU(), nn.Linear(gizli, n_cikis))
    def forward(self, G, gizle_maske=None):
        g, sp = G['g'], G['sp']
        if gizle_maske is not None:
            g = torch.where(gizle_maske, torch.full_like(g, NG), g); sp = torch.where(gizle_maske, torch.full_like(sp, NS), sp)
        d = G['desc'][:, self.sutun]
        d = d + (AYAR['desc_gurultu'] * torch.randn_like(d) if self.training else 0)
        parcalar = [self.ge(g), self.se(sp), d] + ([self.ortam(G)] if self.ortam is not None else [])
        return self.mlp(torch.cat(parcalar, 1))

def monoton(x, a, c, s):
    """Denklem (4): negatif olmayan lojistik karışımı, 0–1 arası, azalmayan. x:[B,24], a,c,s:[B,J]"""
    w = F.softplus(a) + 1e-4
    z = torch.sigmoid((x[..., None] - c[:, None, :]) / s[:, None, :])
    return (z * w[:, None, :]).sum(-1) / w.sum(-1, keepdim=True)

def alan_gor_bul(seri_gor):
    m = torch.zeros(S, dtype=torch.bool, device=DEV); m[ser_site[seri_gor]] = True
    return m

class Gizil(nn.Module):
    """Seri başına gizil başlangıç yarıçapı r0 = softplus(ρ) ve ağaç etkisi u; hiyerarşik önsel cezası"""
    def __init__(self, r0_bas):
        super().__init__()
        rho0 = math.log(math.expm1(r0_bas))
        self.rho = nn.Parameter(torch.full((NSER,), rho0, device=DEV))
        self.mu_g = nn.Parameter(torch.full((NG + 1,), rho0, device=DEV))
        self.u = nn.Parameter(torch.zeros(NTREE, device=DEV))          # ağaç etkisi
        self.v = nn.Parameter(torch.zeros(S, device=DEV))              # alan etkisi (toprak, sıklık vb. kaydedilmemiş farklar)
        # Önsel ölçekleri sabit (zayıf bilgili) hiperparametreler: MAP ile öğrenilen varyans bileşenleri sıfıra çöker.
        # Yeni ağaç/seri için öngörüde kullanılan yayılım ise eğitilmiş ağaçların GÖRGÜL dağılımından alınır.
        self.su_onsel, self.tau_onsel = AYAR['su_onsel'], AYAR['tau_onsel']
        self.register_buffer('su_gorgul', torch.tensor(AYAR['su_onsel'], device=DEV))
        self.register_buffer('tau_gorgul', torch.tensor(AYAR['tau_onsel'], device=DEV))
        self.sv_onsel = AYAR['sv_onsel']
        self.register_buffer('sv_gorgul', torch.tensor(AYAR['sv_onsel'], device=DEV))
        self.register_buffer('ort_u', torch.tensor(0., device=DEV)); self.register_buffer('ort_v', torch.tensor(0., device=DEV))
        self.register_buffer('ort_d', torch.tensor(0., device=DEV))
    def su(self): return self.su_gorgul
    def tau(self): return self.tau_gorgul
    def gorgul_guncelle(self, seri_gor, agac_gor):
        with torch.no_grad():
            g = site_genus[ser_site]
            # Görgül dağılım: ortalama da alınır (ağın sabit terimi ile rastgele etkilerin ortalaması birbirine kayabilir)
            u, d, v = self.u[agac_gor], (self.rho - self.mu_g[g])[seri_gor], self.v[alan_gor_bul(seri_gor)]
            self.su_gorgul.copy_(u.std().clamp_min(0.02)); self.ort_u.copy_(u.mean())
            self.tau_gorgul.copy_(d.std().clamp_min(0.02)); self.ort_d.copy_(d.mean())
            self.sv_gorgul.copy_(v.std().clamp_min(0.02)); self.ort_v.copy_(v.mean())
    def ceza(self, seri_gor, agac_gor):
        """Hiyerarşik önsel; YALNIZ eğitimde verisi olan seri ve ağaçlar üzerinden"""
        g = site_genus[ser_site]
        d = (self.rho - self.mu_g[g])[seri_gor]; u = self.u[agac_gor]; v = self.v[alan_gor_bul(seri_gor)]
        return ((d ** 2).sum() / (2 * self.tau_onsel ** 2) + (u ** 2).sum() / (2 * self.su_onsel ** 2)
                + (v ** 2).sum() / (2 * self.sv_onsel ** 2))
    def gizil_deger(self, ser, ornek=None):
        """Halka başına ρ ve u. ornek=(seri_gor, agac_gor, gizle): görülmemişler için önselden rastgele çekim"""
        rho, u, v = self.rho[ser], self.u[ser_tree[ser]], self.v[ser_site[ser]]
        if ornek is not None:
            seri_gor, agac_gor, gizle, mu_glob = ornek
            g = site_genus[ser_site[ser]]
            mu = torch.where(gizle, mu_glob, self.mu_g[g]) if gizle is not None else self.mu_g[g]
            rho = torch.where(seri_gor[ser], rho, mu + self.ort_d + self.tau() * torch.randn_like(rho))
            u = torch.where(agac_gor[ser_tree[ser]], u, self.ort_u + self.su() * torch.randn_like(u))
            v = torch.where(alan_gor_bul(seri_gor)[ser_site[ser]], v, self.ort_v + self.sv_gorgul * torch.randn_like(v))
        return rho, u + v

class TeoriModeli(nn.Module):
    def __init__(self, J=8, emb=8, gizli=128, ablasyon=None, r0_bas=2.0, m_bas=2/3, vpd=False, m_sabit=None, saf=False, ortam=False):
        super().__init__()
        self.J, self.abl = J, ablasyon
        self.m_logit = math.log(m_bas / (1 - m_bas)); self.vpd = vpd; self.m_sabit = m_sabit
        n = 3 + 6 * J + 24 + 1 + 1          # +1: başlangıç yarıçapı r0 (alan ve takson düzeyinde, ağdan)
        if ablasyon == 'abl_min': n += 3
        if vpd: n += 3 * J                 # VPD yanıtı (üçüncü sınırlayıcı faktör)
        self.Dbaz = torch.linspace(0.2, 3.0, J, device=DEV)
        self.ag = AlanAgi(n, emb, gizli, sutun=[0, 1] if saf else None, ortam=ortam)   # saf: iklim normalleri parametre ağına girmez
        if ablasyon == 'abl_mono':      # monoton yanıt yerine kısıtsız küçük ağ (girdi: x ve alan parametreleri)
            self.serbT = nn.Sequential(nn.Linear(1 + 3 * J, 32), nn.GELU(), nn.Linear(32, 1))
            self.serbM = nn.Sequential(nn.Linear(1 + 3 * J, 32), nn.GELU(), nn.Linear(32, 1))
        self.gz = Gizil(r0_bas)
        self.Tbaz = torch.linspace(-5, 25, J, device=DEV); self.Mbaz = torch.linspace(0.02, 0.6, J, device=DEV)

    def parametreler(self, G, gizle=None):
        if torch.is_tensor(G): G = alan_girdi(G)
        o = self.ag(G, gizle); J = self.J
        p = {}
        p['eta'] = torch.exp(o[:, 0] + math.log(0.7))
        p['Bmax'] = torch.exp(o[:, 1] + math.log(1e4))                         # cm² (başlangıç: r ≈ 56 cm)
        p['m'] = torch.sigmoid(o[:, 2] + self.m_logit)                            # başlangıç varsayılan 2/3
        if self.m_sabit is not None: p['m'] = torch.full_like(p['m'], self.m_sabit)   # profil olabilirlik: m sabit
        i = 3
        p['aT'], p['cT'], p['sT'] = o[:, i:i+J], self.Tbaz + 5 * torch.tanh(o[:, i+J:i+2*J]), 0.5 + 2 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['aM'], p['cM'], p['sM'] = o[:, i:i+J], self.Mbaz + 0.1 * torch.tanh(o[:, i+J:i+2*J]), 0.005 + 0.03 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['omega'] = torch.softmax(o[:, i:i+24], 1); i += 24
        if self.vpd:
            p['aD'], p['cD'], p['sD'] = o[:, i:i+J], self.Dbaz + 0.3 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['log_sigma'] = o[:, i] - 0.3; i += 1
        p['r0'] = F.softplus(o[:, i] + math.log(math.expm1(AYAR['r0_bas_cm']))); i += 1   # cm
        if self.abl == 'abl_min': p['karisim'] = torch.softmax(o[:, i:i+3], 1)
        return p

    def forward(self, idx, gizle=None, ornek=None, G=None):
        G = girdiler(idx) if G is None else G
        X = G['X']; p = self.parametreler(G, gizle)
        ser = G['ser']
        rho_i, u_i = self.gz.gizil_deger(ser, ornek)
        if not AYAR['rastgele_etki']: u_i = torch.zeros_like(u_i)
        r0 = 0.05 if self.abl == 'abl_r0' else (F.softplus(rho_i) if AYAR['rastgele_etki'] else p['r0'])
        r = r0 + G['rp']; B = math.pi * r ** 2                                   # cm²
        # Denklem (2): büyüme potansiyeli
        if self.abl == 'abl_G':
            G = p['eta'] * 30.0
        else:
            x = (B / p['Bmax']).clamp_min(1e-8) ** (1 - p['m'])
            G = p['eta'] * B.clamp_min(1e-6) ** p['m'] * F.softplus(20 * (1 - x)) / 20
        # Denklem (3)–(4): sınırlayıcı faktör
        T, M, gEk = X[..., 0], X[..., 2], X[..., 4]
        if self.abl == 'abl_mono':
            pT = torch.cat([p['aT'], p['cT'], p['sT']], 1); pM = torch.cat([p['aM'], p['cM'], p['sM']], 1)
            gT = torch.sigmoid(self.serbT(torch.cat([T[..., None] / 10, pT[:, None, :].expand(-1, 24, -1)], -1))[..., 0])
            gM = torch.sigmoid(self.serbM(torch.cat([M[..., None] * 5, pM[:, None, :].expand(-1, 24, -1)], -1))[..., 0])
        else:
            gT = monoton(T, p['aT'], p['cT'], p['sT']); gM = monoton(M, p['aM'], p['cM'], p['sM'])
        if self.abl == 'abl_min':
            k = p['karisim'][:, None, :]
            sinir = k[..., 0] * gT + k[..., 1] * gM + k[..., 2] * gT * gM
        else:
            sinir = torch.minimum(gT, gM)
        if self.vpd:     # Liebig: min(g_T, g_M, g_D); g_D = 1 − monoton(VPD) → atmosfer kurudukça büyüme azalır
            gD = 1 - monoton(X[..., 3], p['aD'], p['cD'], p['sD'])
            sinir = torch.minimum(sinir, gD)
        C = (p['omega'] * gEk * sinir).sum(1).clamp_min(1e-6)
        BAI = G * C * torch.exp(u_i)
        w_hat = (torch.sqrt((B + BAI) / math.pi) - r) * 10                         # mm
        mu = torch.log(w_hat.clamp_min(1e-4))
        return mu, p['log_sigma'], dict(gT=gT, gM=gM, C=C, G=G)

class SerbestAg(nn.Module):
    """Kısıtsız karşılaştırma: 24 aylık iklim -> zamansal evrişim; + log yarıçap + alan gömmesi -> log w"""
    def __init__(self, emb=8, gizli=40, r0_bas=2.0, ortam=False):
        super().__init__()
        self.ortam = OrtamGirdi() if ortam else None
        self.conv = nn.Sequential(nn.Conv1d(5, 32, 3), nn.GELU(), nn.Conv1d(32, 32, 3), nn.GELU())
        self.ge = nn.Embedding(NG + 1, emb); self.se = nn.Embedding(NS + 1, emb)
        self.mlp = nn.Sequential(nn.Linear(32 * 20 + 2 * emb + DESC.shape[1] + 1 + (ORTAM_BOYUT if ortam else 0), gizli), nn.GELU(),
                                 nn.Linear(gizli, gizli), nn.GELU(), nn.Linear(gizli, 2))
        self.gz = Gizil(r0_bas)
    def forward(self, idx, gizle=None, ornek=None, G=None):
        G = girdiler(idx) if G is None else G
        X = (G['X'] - oz_ort) / oz_ss
        h = self.conv(X.transpose(1, 2)).flatten(1)
        g, sp = G['g'], G['sp']
        if gizle is not None:
            g = torch.where(gizle, torch.full_like(g, NG), g); sp = torch.where(gizle, torch.full_like(sp, NS), sp)
        lr = torch.log1p(G['rp'])[:, None]
        d = G['desc'] + (AYAR['desc_gurultu'] * torch.randn_like(G['desc']) if self.training else 0)
        parcalar = [h, self.ge(g), self.se(sp), d, lr] + ([self.ortam(G)] if self.ortam is not None else [])
        o = self.mlp(torch.cat(parcalar, 1))
        _, u_i = self.gz.gizil_deger(G['ser'], ornek)
        if not AYAR['rastgele_etki']: u_i = torch.zeros_like(u_i)
        return o[:, 0] - 0.3 + u_i, o[:, 1] - 0.3, {}

def yap(ad):
    """Model adı: taban ad + isteğe bağlı '_ortam' eki (yetişme ortamı girdileri)"""
    ortam = ad.endswith('_ortam'); ad = ad[:-6] if ortam else ad
    T = lambda **k: TeoriModeli(AYAR['J'], AYAR['emb'], AYAR['gizli'], k.pop('abl', None), AYAR['r0_bas_cm'], ortam=ortam, **k).to(DEV)
    if ad == 'serbest': return SerbestAg(AYAR['emb'], r0_bas=AYAR['r0_bas_cm'], ortam=ortam).to(DEV)
    if ad == 'teori': return T()
    if ad == 'teori_vpd': return T(vpd=True)
    if ad == 'teori_saf': return T(vpd=True, saf=True)
    if ad.startswith('teori_mf'):       # profil olabilirlik: teori_mf050 -> m = 0.50 sabit
        return T(m_sabit=int(ad[8:]) / 1000 if len(ad) == 12 else int(ad[8:]) / 100)
    if ad.startswith('teori_m'): return T(m_bas=int(ad[7:]) / 100)
    if ad.startswith('abl_'): return T(abl=ad)
    raise ValueError(ad)

def ag_param_sayisi(m):
    return sum(p.numel() for n, p in m.named_parameters() if not n.startswith('gz.'))

for ad in ['serbest', 'teori', 'teori_vpd', 'serbest_ortam', 'teori_ortam', 'teori_vpd_ortam']:
    print(f'{ad:16s} eğitilebilir ağ parametresi: {ag_param_sayisi(yap(ad)):,}')

In [ ]:
# --- Eğitim ---
LOG_Y_SIFIR = math.log(AYAR['sifir_mm'])
def hedef(idx):
    w = R_w[idx]
    return torch.where(w > 0, torch.log(w.clamp_min(1e-6)), torch.full_like(w, LOG_Y_SIFIR))

LOG_ESIK = math.log(0.01)            # ölçüm çözünürlüğü: sıfır halka = "0,01 mm'den küçük" (sansürlü gözlem)
def nll(mu, ls, y):
    sifir = y <= LOG_Y_SIFIR + 1e-6
    tam = 0.5 * ((y - mu) / torch.exp(ls)) ** 2 + ls + 0.9189385
    sans = -torch.special.log_ndtr((LOG_ESIK - mu) / torch.exp(ls))
    return torch.where(sifir, sans, tam).mean()

@torch.no_grad()
def tahmin(model, idx, seri_gor, agac_gor, gizle_alan=False, K=8, bs=131072):
    """Görülmüş ağaçlar: koşullu öngörü. Görülmemiş ağaçlar (yeni alan/tür): ρ ve u önselden K kez çekilir;
    öngörü = karışımın ortalaması, varyans = ort(σ²) + var(μ)  (log ölçeğinde)"""
    model.eval(); MU, SG = [], []
    g_gor = torch.unique(site_genus[ser_site[seri_gor.nonzero().squeeze(1)]])
    mu_glob = model.gz.mu_g[g_gor].mean()
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            b = idx[i:i+bs]
            gz = torch.ones(len(b), dtype=torch.bool, device=DEV) if gizle_alan else None
            yeni = ~agac_gor[ser_tree[R_ser[b]]] | ~seri_gor[R_ser[b]]
            if not AYAR['rastgele_etki']: yeni = torch.zeros_like(yeni)
            if not yeni.any():
                mu, ls, _ = model(b, gz); MU.append(mu); SG.append(torch.exp(ls)); continue
            ms, vs = [], []
            for k in range(K):
                mu, ls, _ = model(b, gz, ornek=(seri_gor, agac_gor, gz, mu_glob)); ms.append(mu); vs.append(torch.exp(2 * ls))
            ms, vs = torch.stack(ms), torch.stack(vs)
            MU.append(ms.mean(0)); SG.append(torch.sqrt(vs.mean(0) + ms.var(0, unbiased=False)))
    model.train()
    return torch.cat(MU), torch.cat(SG)

def kat_gor_ayarla(egt):
    """Eğitim halkalarının alanlarında görülen biyom / ekolojik bölge / Köppen sınıfları"""
    sg = torch.unique(ser_site[R_ser[egt]]); KAT_GOR.clear()
    for anahtar, dizi, n in [('b', site_biyom, NB), ('e', site_eko, NE), ('k', site_kg, NK)]:
        m = torch.zeros(n + 1, dtype=torch.bool, device=DEV); m[dizi[sg]] = True; KAT_GOR[anahtar] = m

def egit(bolunme_ad, model_ad, tohum):
    is_ad = f'{bolunme_ad.replace(":", "-")}__{model_ad}__s{tohum}'
    klasor = os.path.join(SONUC, is_ad); os.makedirs(klasor, exist_ok=True)
    if os.path.exists(os.path.join(klasor, 'bitti.json')):
        print('atlandı (bitmiş):', is_ad); return
    torch.manual_seed(tohum); np.random.seed(tohum)
    egt, dog, test, gizle = bolunme(bolunme_ad)
    kat_gor_ayarla(egt)
    if AYAR['hizli_deneme']:
        egt, dog, test = egt[torch.randperm(len(egt), device=DEV)[:200000]], dog[:20000], test[:20000]
    model = yap(model_ad)
    gizil = [p for n, p in model.named_parameters() if n.startswith('gz.')]
    ag = [p for n, p in model.named_parameters() if not n.startswith('gz.')]
    opt = torch.optim.AdamW([{'params': ag, 'lr': AYAR['lr'], 'weight_decay': AYAR['wd']},
                             {'params': gizil, 'lr': AYAR['lr_gizil'] if AYAR['rastgele_etki'] else 0.0, 'weight_decay': 0.0}])
    kayit = os.path.join(klasor, 'son.pt'); bas_ep, en_iyi, sabir, gecmis = 0, float('inf'), 0, []
    if os.path.exists(kayit):
        ck = torch.load(kayit, map_location=DEV)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        bas_ep, en_iyi, sabir, gecmis = ck['epoch'] + 1, ck['en_iyi'], ck['sabir'], ck['gecmis']
        print(f'{is_ad}: {bas_ep}. epoch\'tan devam')
    n_egt = len(egt)
    agac_gor = torch.zeros(NTREE, dtype=torch.bool, device=DEV); agac_gor[ser_tree[R_ser[egt]]] = True
    seri_gor = torch.zeros(NSER, dtype=torch.bool, device=DEV); seri_gor[R_ser[egt]] = True
    for ep in range(bas_ep, AYAR['max_epoch']):
        t0 = time.time(); kar = egt[torch.randperm(n_egt, device=DEV)]; top = 0.
        for i in range(0, n_egt, AYAR['batch']):
            b = kar[i:i+AYAR['batch']]
            gz = (torch.rand(len(b), device=DEV) < AYAR['takson_dusur'])
            mu, ls, _ = model(b, gz)
            kayip_v = nll(mu, ls, hedef(b))
            kayip = kayip_v + (model.gz.ceza(seri_gor, agac_gor) / n_egt if AYAR['rastgele_etki'] else 0.)
            opt.zero_grad(set_to_none=True); kayip.backward()
            torch.nn.utils.clip_grad_norm_(ag, 5.0); opt.step(); top += kayip_v.item() * len(b)
        model.gz.gorgul_guncelle(seri_gor, agac_gor)
        mu, sg = tahmin(model, dog, seri_gor, agac_gor)
        yd = hedef(dog); v = nll(mu, torch.log(sg), yd).item()
        nz = yd > LOG_Y_SIFIR + 1e-6; v_rmse = torch.sqrt(((mu - yd)[nz] ** 2).mean()).item(); v_sg = torch.sqrt((sg ** 2).mean()).item()
        gecmis.append(dict(epoch=ep, egitim=top / n_egt, dogrulama=v, sn=time.time() - t0))
        ek = (f'  σ_alan {model.gz.sv_gorgul.item():.2f}  σ_ağaç {model.gz.su().item():.2f}' if AYAR['rastgele_etki'] else '')
        print(f'{is_ad} ep {ep:2d}  eğitim {top/n_egt:.4f}  doğrulama {v:.4f} (RMSE {v_rmse:.3f}, öngörü σ {v_sg:.3f}){ek}  ({time.time()-t0:.0f} sn)')
        if v < en_iyi - 1e-4:
            en_iyi, sabir = v, 0; torch.save(model.state_dict(), os.path.join(klasor, 'en_iyi.pt'))
        else:
            sabir += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), epoch=ep, en_iyi=en_iyi, sabir=sabir, gecmis=gecmis), kayit)
        if sabir >= AYAR['sabir']: break
    # Test tahminleri (en iyi model)
    model.load_state_dict(torch.load(os.path.join(klasor, 'en_iyi.pt'), map_location=DEV))
    mu, sg = tahmin(model, test, seri_gor, agac_gor, gizle_alan=gizle)
    y = hedef(test)
    np.savez_compressed(os.path.join(klasor, 'test_tahmin.npz'), idx=test.cpu().numpy().astype(np.int32),
                        mu=mu.cpu().numpy().astype(np.float16), sigma=sg.cpu().numpy().astype(np.float16))
    nz = y > LOG_Y_SIFIR + 1e-6         # nokta ölçütleri sıfır olmayan halkalarda; NLL sansürlü olanları da içerir
    ozet = dict(is_ad=is_ad, n_egitim=n_egt, n_test=len(test), rmse_log=float(torch.sqrt(((mu - y)[nz] ** 2).mean())),
                bias_log=float((mu - y)[nz].mean()), nll=float(nll(mu, torch.log(sg), y)),
                kapsama90=float(((y - mu).abs() < 1.645 * sg)[nz].float().mean()), en_iyi_dogrulama=en_iyi,
                ag_param=ag_param_sayisi(model), gecmis=gecmis, ayar=AYAR)
    if not model_ad.startswith('serbest'):           # alan parametreleri (m, eşikler, ω) — şekiller için
        with torch.no_grad():
            p = model.parametreler(torch.arange(S, device=DEV), torch.full((S,), gizle, device=DEV) & t(site_takson, torch.bool))
        np.savez_compressed(os.path.join(klasor, 'alan_param.npz'), **{k: v.cpu().numpy() for k, v in p.items()},
                            r0_cm=F.softplus(model.gz.rho).detach().cpu().numpy(), u=model.gz.u.detach().cpu().numpy())
    json.dump(ozet, open(os.path.join(klasor, 'bitti.json'), 'w'), indent=1)
    os.remove(kayit)
    print(f'BİTTİ {is_ad}: test RMSE(log) {ozet["rmse_log"]:.3f}, kapsama %90 → {ozet["kapsama90"]:.3f}')

In [ ]:
# --- İş kuyruğu (Analiz Planı, 21 Ekim 2026) ---
BOLUNMELER = ['zaman', 'takson', 'bolge:North America', 'bolge:Europe', 'bolge:Asia', 'bolge:Southern Hemisphere', 'iklim:sicak', 'iklim:kurak']
MODELLER = ['serbest', 'teori', 'teori_vpd', 'serbest_ortam', 'teori_ortam', 'teori_vpd_ortam']
KUYRUK = []
for tohum in range(5):                                   # 5 üyeli topluluk; tohum dışta: kuyruk yarıda kalsa da dengeli sonuç
    for b in BOLUNMELER:
        for m in MODELLER: KUYRUK.append((b, m, tohum))
for b in ['zaman', 'takson', 'bolge:Southern Hemisphere', 'iklim:kurak']:     # ablasyonlar (teori + ortam)
    for m in ['abl_G', 'abl_min', 'abl_mono', 'abl_r0']: KUYRUK.append((b, m + '_ortam', 0))
for mf in ['teori_mf030', 'teori_mf040', 'teori_mf050', 'teori_mf060', 'teori_mf0667', 'teori_mf075', 'teori_mf085']:
    KUYRUK.append(('zaman', mf + '_ortam', 0))       # m profil olabilirliği
print(len(KUYRUK), 'iş;', sum(os.path.exists(os.path.join(SONUC, f'{b.replace(":", "-")}__{m}__s{s}', 'bitti.json')) for b, m, s in KUYRUK), 'tanesi bitmiş')

In [ ]:
for b, m, s in KUYRUK:
    egit(b, m, s)
print('==== KUYRUK TAMAM ====')

In [ ]:
# --- Serbest gidişli öngörü (zamansal test): ağaç 1960'tan sonra KENDİ tahmin ettiği halkalarla büyütülür ---
# Gözlenen yarıçap yalnız 1960 sonunda bilinir; sonraki her yıl yarıçap = önceki yarıçap + modelin öngördüğü halka (medyan).
@torch.no_grad()
def serbest_gidis(model_ad, tohum):
    global R_rp
    klasor = os.path.join(SONUC, f'zaman__{model_ad}__s{tohum}')
    hedef_dosya = os.path.join(klasor, 'serbest_gidis.json')
    if not os.path.exists(os.path.join(klasor, 'bitti.json')) or os.path.exists(hedef_dosya): return
    egt_z, _, test, _ = bolunme('zaman'); kat_gor_ayarla(egt_z)
    model = yap(model_ad); model.load_state_dict(torch.load(os.path.join(klasor, 'en_iyi.pt'), map_location=DEV)); model.eval()
    asil = R_rp.clone(); R_rp = asil.clone()
    try:
        yil_t = R_yil[test]; mu_hepsi = torch.empty(len(test), device=DEV); sg_hepsi = torch.empty(len(test), device=DEV)
        for y in range(1961, YIL0 + NY):
            k = (yil_t == y).nonzero().squeeze(1)
            if len(k) == 0: continue
            sel = test[k]; mu, ls, _ = model(sel); mu_hepsi[k] = mu; sg_hepsi[k] = torch.exp(ls)
            nxt = (sel + 1).clamp_max(NR - 1)
            ok = (R_ser[nxt] == R_ser[sel]) & (R_yil[nxt] == y + 1)
            R_rp[nxt[ok]] = R_rp[sel[ok]] + torch.exp(mu[ok]) / 10          # mm -> cm
        y_obs = hedef(test); nz = y_obs > LOG_Y_SIFIR + 1e-6
        # Dönem sonunda birikmiş yarıçap artımı: öngörülen / gözlenen
        ser = R_ser[test]
        art_obs = torch.zeros(NSER, device=DEV).index_add_(0, ser, R_w[test] / 10)
        art_pre = torch.zeros(NSER, device=DEV).index_add_(0, ser, torch.exp(mu_hepsi) / 10)
        var = art_obs > 0
        lr = torch.log(art_pre[var] / art_obs[var])
        ozet = dict(model=model_ad, tohum=tohum, rmse_log_halka=float(torch.sqrt(((mu_hepsi - y_obs)[nz] ** 2).mean())),
                    birikimli_log_oran_rmse=float(torch.sqrt((lr ** 2).mean())), birikimli_log_oran_bias=float(lr.mean()),
                    n_seri=int(var.sum()))
        json.dump(ozet, open(hedef_dosya, 'w'), indent=1); print('serbest gidiş', ozet)
    finally:
        R_rp = asil

for tohum in range(5):
    for m in MODELLER: serbest_gidis(m, tohum)
for m in ['abl_G', 'abl_min', 'abl_mono', 'abl_r0']: serbest_gidis(m + '_ortam', 0)
print('==== SERBEST GİDİŞ TAMAM ====')

## Model içi analizler (Analiz Planı 4): teori tutarlılığı, SHAP, kurak alan teşhisi
Eğitilmiş modelleri (tohum 0) yükler; sonuçlar `SONUC_SON/MODEL_ANALIZ/` klasörüne CSV olarak yazılır. GPU'da ~10–20 dk.

In [ ]:
MA = os.path.join(SONUC, 'MODEL_ANALIZ'); os.makedirs(MA, exist_ok=True)
import pandas as pd
def yukle(bolunme_ad, model_ad, tohum=0):
    kl = os.path.join(SONUC, f'{bolunme_ad.replace(":", "-")}__{model_ad}__s{tohum}')
    if not os.path.exists(os.path.join(kl, 'en_iyi.pt')): return None, None
    egt, _, test, gizle = bolunme(bolunme_ad); kat_gor_ayarla(egt)
    m = yap(model_ad); m.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV)); m.eval()
    return m, (egt, test, gizle)

@torch.no_grad()
def tahmin_G(model, G, gizle=False, bs=65536):
    n = len(G['rp']); out = []
    for i in range(0, n, bs):
        Gi = {k: v[i:i+bs] for k, v in G.items()}
        gz = torch.ones(len(Gi['rp']), dtype=torch.bool, device=DEV) if gizle else None
        out.append(model(None, gz, G=Gi)[0])
    return torch.cat(out)

# --- 1. Teori tutarlılığı (ihlal analizi): zamansal test, 50.000 halka ---
gen = torch.Generator(device='cpu').manual_seed(0)
ihlal = []
for m_ad in MODELLER:
    model, b = yukle('zaman', m_ad)
    if model is None: continue
    test = b[1]; sec = test[torch.randperm(len(test), generator=gen)[:50000].to(DEV)]
    G0 = girdiler(sec); y0 = tahmin_G(model, G0)
    G1 = dict(G0); X = G0['X'].clone(); X[..., 2] = (X[..., 2] + 0.05).clamp_max(0.76); G1['X'] = X      # tüm aylarda toprak nemi +0,05
    dM = tahmin_G(model, G1) - y0
    G2 = dict(G0); X = G0['X'].clone(); X[..., 1] = torch.log1p(torch.expm1(X[..., 1]) * 1.2); G2['X'] = X   # yağış +%20 (yalnız serbest ağ kullanır)
    dP = tahmin_G(model, G2) - y0
    G3 = dict(G0); X = G0['X'].clone(); X[..., 3] = X[..., 3] + 0.2; G3['X'] = X                         # VPD +0,2 kPa
    dD = tahmin_G(model, G3) - y0
    s_id = ser_site[R_ser[sec]].cpu().numpy()
    df = pd.DataFrame(dict(s=s_id, dM=dM.cpu().numpy(), dP=dP.cpu().numpy(), dD=dD.cpu().numpy()))
    alan_ort = df.groupby('s')[['dM', 'dP', 'dD']].median()
    ihlal.append(dict(model=m_ad, n_halka=len(df),
        nem_artinca_azalan_halka=float((df.dM < -1e-3).mean()), nem_artinca_azalan_alan=float((alan_ort.dM < -1e-3).mean()),
        yagis_artinca_azalan_halka=float((df.dP < -1e-3).mean()), yagis_artinca_azalan_alan=float((alan_ort.dP < -1e-3).mean()),
        vpd_artinca_artan_halka=float((df.dD > 1e-3).mean()), vpd_artinca_artan_alan=float((alan_ort.dD > 1e-3).mean())))
    print('ihlal', ihlal[-1])
pd.DataFrame(ihlal).to_csv(os.path.join(MA, 'T9_teori_ihlal.csv'), index=False)

In [ ]:
# --- 2. SHAP (grup düzeyinde, girişimsel Shapley; permütasyon örneklemesi) ---
# Gruplar: ağaç büyüklüğü, aylık iklim, konum (enlem+yükselti), iklim normalleri, toprak, topoğrafya, ekolojik sınıf, takson.
# Her halka için arka plan olarak rastgele bir eğitim halkası kullanılır; grubu "yokken" o halkanın değerleri girer.
GRUPLAR = {'büyüklük': ['rp'], 'iklim (aylık)': ['X'], 'konum': [('desc', [0, 1])], 'iklim normali': [('desc', [2, 3, 4, 5, 6, 7])],
           'toprak': [('env', list(range(3, 11)))], 'topoğrafya': [('env', [0, 1, 2])], 'ekolojik sınıf': ['b', 'e', 'k'], 'takson': ['g', 'sp']}
def karistir(Gx, Gb, gruplar_var):
    """gruplar_var içindeki gruplar x'ten, diğerleri arka plandan"""
    G = {k: v.clone() for k, v in Gb.items()}; G['ser'] = Gx['ser']
    for gr in gruplar_var:
        for a in GRUPLAR[gr]:
            if isinstance(a, tuple): G[a[0]][:, a[1]] = Gx[a[0]][:, a[1]]
            else: G[a] = Gx[a]
    return G

@torch.no_grad()
def grup_shap(model, sec, arka, n_perm=24):
    Gx, Gb = girdiler(sec), girdiler(arka); ad = list(GRUPLAR); phi = torch.zeros(len(sec), len(ad), device=DEV)
    for _ in range(n_perm):
        sira = np.random.permutation(len(ad)); var = []; once = tahmin_G(model, karistir(Gx, Gb, var))
        for j in sira:
            var.append(ad[j]); simdi = tahmin_G(model, karistir(Gx, Gb, var)); phi[:, j] += simdi - once; once = simdi
    return (phi / n_perm).cpu().numpy(), ad

np.random.seed(0); shap_ozet = []
for m_ad in ['serbest_ortam', 'teori_ortam', 'teori_vpd_ortam']:
    model, b = yukle('zaman', m_ad)
    if model is None: continue
    egt, test = b[0], b[1]
    sec = test[torch.randperm(len(test), generator=gen)[:20000].to(DEV)]; arka = egt[torch.randperm(len(egt), generator=gen)[:20000].to(DEV)]
    phi, ad = grup_shap(model, sec, arka)
    np.save(os.path.join(MA, f'shap_{m_ad}.npy'), phi.astype(np.float32))
    for j, g in enumerate(ad): shap_ozet.append(dict(model=m_ad, grup=g, ort_mutlak_shap=float(np.abs(phi[:, j]).mean())))
    print(m_ad, {g: round(float(np.abs(phi[:, j]).mean()), 4) for j, g in enumerate(ad)})
pd.DataFrame(shap_ozet).to_csv(os.path.join(MA, 'T10_shap_grup.csv'), index=False)

# SHAP: teori modelinin parametre ağı — η, B_max, m'yi hangi alan değişkenleri belirliyor? (alan düzeyinde)
model, b = yukle('zaman', 'teori_ortam')
if model is not None:
    ALAN_GR = {k: v for k, v in GRUPLAR.items() if k not in ('büyüklük', 'iklim (aylık)')}
    @torch.no_grad()
    def par(G): p = model.parametreler(G); return torch.stack([torch.log(p['eta']), torch.log(p['Bmax']), p['m']], 1)
    s_all = torch.arange(S, device=DEV); s_bg = s_all[torch.randperm(S, generator=gen).to(DEV)]
    Gx, Gb = alan_girdi(s_all), alan_girdi(s_bg); ad = list(ALAN_GR); phi = torch.zeros(S, len(ad), 3, device=DEV)
    def kar(var):
        G = {k: v.clone() for k, v in Gb.items()}
        for gr in var:
            for a in ALAN_GR[gr]:
                if isinstance(a, tuple): G[a[0]][:, a[1]] = Gx[a[0]][:, a[1]]
                else: G[a] = Gx[a]
        return G
    for _ in range(48):
        sira = np.random.permutation(len(ad)); var = []; once = par(kar(var))
        for j in sira:
            var.append(ad[j]); simdi = par(kar(var)); phi[:, j] += simdi - once; once = simdi
    phi = (phi / 48).cpu().numpy()
    pd.DataFrame([dict(parametre=pn, grup=g, ort_mutlak_shap=float(np.abs(phi[:, j, q]).mean()))
                  for q, pn in enumerate(['log_eta', 'log_Bmax', 'm']) for j, g in enumerate(ad)]).to_csv(os.path.join(MA, 'T11_shap_parametre.csv'), index=False)
    print('parametre SHAP kaydedildi')

In [ ]:
# --- 3. Kurak alan teşhisi: teori modelinin iç bileşenleri, eğitim ve kurak test alanlarında ---
teshis = []
for m_ad in ['teori', 'teori_ortam', 'teori_vpd_ortam']:
    model, b = yukle('iklim:kurak', m_ad)
    if model is None: continue
    egt, test = b[0], b[1]
    for ad_k, kume in [('egitim', egt), ('kurak_test', test)]:
        sec = kume[torch.randperm(len(kume), generator=gen)[:50000].to(DEV)]
        with torch.no_grad():
            mu, ls, ek = model(sec)
        y = hedef(sec); nz = y > LOG_Y_SIFIR + 1e-6
        nem_sinir = (ek['gM'] < ek['gT']).float().mean(1)
        s_id = ser_site[R_ser[sec]]
        teshis.append(dict(model=m_ad, kume=ad_k, bias_log=float((mu - y)[nz].mean()), ort_C=float(ek['C'].mean()), ort_G=float(ek['G'].mean()),
                           ort_gM=float(ek['gM'].mean()), ort_gT=float(ek['gT'].mean()), nem_sinirli_ay_orani=float(nem_sinir.mean()),
                           kuraklik_P_PET=float(t(desc[:, 5], torch.float32)[s_id].mean())))
        print(teshis[-1])
pd.DataFrame(teshis).to_csv(os.path.join(MA, 'T12_kurak_teshis.csv'), index=False)
print('==== MODEL İÇİ ANALİZLER TAMAM ====')